In [2]:
import pandas as pd
import numpy as np

In [3]:
raw = pd.read_csv('../data/clean/d1_results_unique.csv')
raw

,season,date,event,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,result,result_type,score,match_duration
0,2013/2014,2013-11-01,Oklahoma Gold Classic,125,Mike Soria,37251,Buffalo,Max Soria,12190,Buffalo,L,MFOR,0 - 0,0:00
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,125,Hunter Weber,13273,North Dakota State,Barlow McGhee,13251,Missouri,L,DEC,2 - 5,7:00
2,2013/2014,2013-11-01,Oklahoma Gold Classic,125,David Terao,12068,American,Max Soria,12190,Buffalo,W,DEC,12 - 6,7:00
3,2013/2014,2013-11-01,Oklahoma Gold Classic,125,Sean Boylan,37235,Bloomsburg,Max Soria,12190,Buffalo,L,DEC,2 - 4,7:00
4,2013/2014,2013-11-01,Oklahoma Gold Classic,125,Hunter Wood,13327,Army West Point,Max Soria,12190,Buffalo,L,DEC,3 - 9,7:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263417,2024/2025,2025-03-20,NCAA Championships,285,Dayton Pitzer,72598,Pittsburgh,Daniel Bucknavich,62488,Cleveland State,W,DEC,6 - 2,7:00
263418,2024/2025,2025-03-20,NCAA Championships,285,Josh Heindselman,57311,Michigan,Isaac Trumble,67131,NC State,L,MD,0 - 9,7:00
263419,2024/2025,2025-03-20,NCAA Championships,285,Josh Heindselman,57311,Michigan,Brady Colbert,78625,Army West Point,W,TF,17 - 2,4:29
263420,2024/2025,2025-03-20,NCAA Championships,285,Wyatt Hendrickson,56694,Oklahoma State,Isaac Trumble,67131,NC State,W,FALL,FALL,2:15


In [4]:
def duration_to_seconds(duration_str):
    try:
        minutes, seconds = map(int, duration_str.split(":"))
        return minutes * 60 + seconds
    except:
        return None

def double_matches(df):
    # Create copy for reversal
    df_b = df.copy()

    # Reverse wrestler and opponent in copy df
    df_b['wrestler_id'] = df['opponent_id']
    df_b['opponent_id'] = df['wrestler_id']
    df_b['wrestler'] = df['opponent']
    df_b['opponent'] = df['wrestler']
    df_b['wrestler_school'] = df['opponent_school']
    df_b['opponent_school'] = df['wrestler_school']
    df_b['wrestler_score'] = df['opponent_score']
    df_b['opponent_score'] = df['wrestler_score']
    df_b['result'] = np.where(df['result'] == 'W', 'L', 'W')

    # Combine and return df and copy
    result = pd.concat([df, df_b])
    return result.sort_values(['date', 'wrestler_id']).reset_index()

def create_base_features(df):
    df[['wrestler_score', 'opponent_score']] = df['score'].str.split(' - ', expand=True)
    df.loc[df['result_type'] == 'FALL', ['wrestler_score', 'opponent_score']] = 0
    df = df.drop(['score'], axis=1)
    
    df['is_dual_meet'] = [1 if 'Dual' in x else 0 for x in df['event']]

    df = df[~df['result_type'].isin(['MFOR', 'INJ', 'CMFF', 'DQ', 'DEF', 'FOR'])].copy()

    # Convert to seconds
    df["duration_seconds"] = df["match_duration"].apply(duration_to_seconds)
    # Create is_overtime column (True if > 7 minutes)
    df["is_overtime"] = (df["duration_seconds"] > 420).astype(int)
    df = df.drop('match_duration', axis=1)

    df['wrestler_score'] = pd.to_numeric(df['wrestler_score'], errors='coerce')
    df['opponent_score'] = pd.to_numeric(df['opponent_score'], errors='coerce')

    df = double_matches(df)

    df['is_win'] = (df['result'] == 'W').astype(int)
    df['point_differential'] = df['wrestler_score'] - df['opponent_score']

    df = df.sort_values(['wrestler_id', 'date']).reset_index(drop=True)

    return df

In [5]:
df = create_base_features(raw).drop(columns=['index'])

In [6]:
df

,season,date,event,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,result,result_type,wrestler_score,opponent_score,is_dual_meet,duration_seconds,is_overtime,is_win,point_differential
0,2013/2014,2013-11-22,Baker (Kan) - Iowa Dual,165,Nick Moore,11454,Iowa,Jakob Price,37118,Baker,W,FALL,0,0,1,175,0,1,0
1,2013/2014,2013-11-22,Iowa Central Community College - Iowa Dual,165,Nick Moore,11454,Iowa,Corbin Farrell,11456,Iowa Central Community College,W,FALL,0,0,1,104,0,1,0
2,2013/2014,2013-12-01,Iowa - Iowa State Dual,165,Nick Moore,11454,Iowa,Michael Moreno,11457,Iowa State,W,DEC,3,1,1,420,0,1,2
3,2013/2014,2013-12-05,Iowa - Edinboro Dual,165,Nick Moore,11454,Iowa,Zach Towers,11458,Edinboro,W,DEC,11,4,1,420,0,1,7
4,2013/2014,2013-12-12,Iowa - Buffalo Dual,165,Nick Moore,11454,Iowa,Wally Maziarz,11459,Buffalo,W,MD,13,4,1,420,0,1,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506467,2024/2025,2025-03-02,Zingo Nationals,197,Tommy Renna,101228,Unattached,Kevin Taylor,93979,Sacred Heart,L,DEC,0,7,0,420,0,0,-7
506468,2022/2023,2022-11-13,Jonathan Kaloust Bearcat Open,197,Jude Correa,101285,High School,Lucas Cochran,71705,Penn State,L,DEC,1,5,0,420,0,0,-4
506469,2022/2023,2022-11-13,Jonathan Kaloust Bearcat Open,197,Jude Correa,101285,High School,Levko Higgins,50870,Penn State,W,FALL,0,0,0,101,0,1,0
506470,2022/2023,2022-11-13,Jonathan Kaloust Bearcat Open,197,Jude Correa,101285,High School,Franklin Cruz,72182,Northern Colorado,W,FALL,0,0,0,135,0,1,0


In [7]:
def calculate_streak(wins):
    """Calculate current win/loss streak for each match"""
    streaks = [0]
    current_streak = 0
    
    for i in range(1, len(wins)):
        # FIXED: Look at the actual previous match result
        prev_is_win = wins.iloc[i-1]
        
        if i == 1:
            # First streak is just 1 (win or loss)
            current_streak = 1
        else:
            # Check if previous match had same result as match before that
            if wins.iloc[i-2] == prev_is_win:
                current_streak += 1
            else:
                current_streak = 1
        
        # Positive for win streaks, negative for loss streaks
        streak_value = current_streak if prev_is_win == 1 else -1 * current_streak
        streaks.append(streak_value)
    
    return streaks

In [8]:
test = df[df['wrestler'] == 'Jacori Teemer'].head(15)
streaks = test.groupby('wrestler_id')['is_win'].transform(
    lambda x: calculate_streak(x.reset_index(drop=True))
)
test['streak'] = streaks
test

,season,date,event,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,result,result_type,wrestler_score,opponent_score,is_dual_meet,duration_seconds,is_overtime,is_win,point_differential,streak
268337,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Brayton Lee,52079,Minnesota,W,DEC,9,7,0,420,0,1,2,0
268338,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Sammy Sasso,51143,Ohio State,W,DEC,8,6,0,420,0,1,2,1
268339,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Matt Kolodzik,45071,Princeton,L,SV-1,4,6,0,540,1,0,-2,2
268340,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Patricio Lugo,39919,Iowa,L,DEC,3,10,0,420,0,0,-7,-1
268341,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Alfred Bannister,19997,Maryland,W,DEC,9,5,0,420,0,1,4,-2
268342,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Khristian Olivas,40318,Fresno State,W,MD,11,2,0,420,0,1,9,1
268343,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Jake Benner,50617,Rutgers,W,DEC,8,5,0,420,0,1,3,2
268344,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Tyler Eischens,51205,Stanford,W,DEC,3,2,0,420,0,1,1,3
268345,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Justin McCoy,50946,Virginia,W,DEC,7,5,0,420,0,1,2,4
268346,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Seth Hogue,44668,West Virginia,W,FALL,0,0,0,257,0,1,0,5


In [9]:
# Historical Performance Features
for window in [3, 5, 10, 15]:
        df[f'win_rate_last_{window}'] = df.groupby('wrestler_id')['is_win'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
        )
    
# FIXED: Win/loss streaks
streaks = df.groupby('wrestler_id')['is_win'].transform(
    lambda x: calculate_streak(x.reset_index(drop=True))
)
df['streak'] = streaks

# FIXED: Career totals (up to current match) - ensure first match has 0s
df['career_wins'] = df.groupby('wrestler_id')['is_win'].transform(
    lambda x: x.cumsum().shift(1).fillna(0)
)
df['career_losses'] = df.groupby('wrestler_id')['is_win'].transform(
    lambda x: (1 - x).cumsum().shift(1).fillna(0)
)
df['career_matches'] = df['career_wins'] + df['career_losses']

# FIXED: Seasonal performance - ensure first match of season has 0 wins
df['season_wins'] = df.groupby(['wrestler_id', 'season'])['is_win'].transform(
    lambda x: x.cumsum().shift(1).fillna(0)
)
df['season_matches'] = df.groupby(['wrestler_id', 'season']).cumcount()
df['season_win_rate'] = np.where(df['season_matches'] > 0, 
                                df['season_wins'] / df['season_matches'], 0)

In [10]:
df[df['wrestler'] == 'Jacori Teemer'].head(15)[['career_wins', 'career_losses', 'career_matches']]

,career_wins,career_losses,career_matches
268337,0.0,0.0,0.0
268338,1.0,0.0,1.0
268339,2.0,0.0,2.0
268340,2.0,1.0,3.0
268341,2.0,2.0,4.0
268342,3.0,2.0,5.0
268343,4.0,2.0,6.0
268344,5.0,2.0,7.0
268345,6.0,2.0,8.0
268346,7.0,2.0,9.0


In [11]:
# Match Quality Features
bonus_results = ['FALL', 'TF', 'MD']
df['bonus_win'] = ((df['result'] == 'W') & 
                        (df['result_type'].isin(bonus_results))).astype(int)

# Close matches (within 3 points)
df['close_match'] = (abs(df['point_differential']) <= 3).astype(int)

# Rolling percentages for outcome quality
for window in [5, 10]:
    df[f'bonus_win_rate_last_{window}'] = df.groupby('wrestler_id')['bonus_win'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )
    
    # Close match win rate
    df['close_match_win'] = df['is_win'] * df['close_match']
    df[f'close_match_win_rate_last_{window}'] = df.groupby('wrestler_id')['close_match_win'].transform(
        lambda x: x.rolling(window=window, min_periods=1).sum().shift(1)
    ) / df.groupby('wrestler_id')['close_match'].transform(
        lambda x: x.rolling(window=window, min_periods=1).sum().shift(1)
    ).replace(0, np.nan)
    df[f'close_match_win_rate_last_{window}'] = df[f'close_match_win_rate_last_{window}'].fillna(0)

In [12]:
df[df['wrestler'] == 'Jacori Teemer'].head(15)

,season,date,event,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,...,season_wins,season_matches,season_win_rate,bonus_win,close_match,bonus_win_rate_last_5,close_match_win,close_match_win_rate_last_5,bonus_win_rate_last_10,close_match_win_rate_last_10
268337,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Brayton Lee,52079,Minnesota,...,0.0,0,0.000000,0,1,NaN,1,0.000000,NaN,0.000000
268338,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Sammy Sasso,51143,Ohio State,...,1.0,1,1.000000,0,1,0.0,1,1.000000,0.000000,1.000000
268339,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Matt Kolodzik,45071,Princeton,...,2.0,2,1.000000,0,1,0.0,0,1.000000,0.000000,1.000000
268340,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Patricio Lugo,39919,Iowa,...,2.0,3,0.666667,0,0,0.0,0,0.666667,0.000000,0.666667
268341,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Alfred Bannister,19997,Maryland,...,2.0,4,0.500000,0,0,0.0,0,0.666667,0.000000,0.666667
268342,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Khristian Olivas,40318,Fresno State,...,3.0,5,0.600000,1,0,0.0,0,0.666667,0.000000,0.666667
268343,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Jake Benner,50617,Rutgers,...,4.0,6,0.666667,0,1,0.2,1,0.500000,0.166667,0.666667
268344,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Tyler Eischens,51205,Stanford,...,5.0,7,0.714286,0,1,0.2,1,0.500000,0.142857,0.750000
268345,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Justin McCoy,50946,Virginia,...,6.0,8,0.750000,0,1,0.2,1,1.000000,0.125000,0.800000
268346,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Seth Hogue,44668,West Virginia,...,7.0,9,0.777778,1,1,0.2,1,1.000000,0.111111,0.833333


In [13]:
# Scoring and Competition Features
# Rolling scoring averages
for window in [3, 5, 10]:
    df[f'avg_points_scored_last_{window}'] = df.groupby('wrestler_id')['wrestler_score'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )
    df[f'avg_points_allowed_last_{window}'] = df.groupby('wrestler_id')['opponent_score'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )
    df[f'avg_point_differential_last_{window}'] = (df[f'avg_points_scored_last_{window}'] - 
                                                    df[f'avg_points_allowed_last_{window}'])

# Overtime frequency
for window in [5, 10]:
    df[f'overtime_rate_last_{window}'] = df.groupby('wrestler_id')['is_overtime'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )

# Match duration averages
for window in [5, 10]:
    df[f'avg_duration_last_{window}'] = df.groupby('wrestler_id')['duration_seconds'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )

# Competition format performance
# Dual meet performance
df['dual_meet_wins'] = df.groupby('wrestler_id')[['is_win', 'is_dual_meet']].apply(
    lambda x: (x['is_win'] * x['is_dual_meet']).cumsum().shift(1).fillna(0)
).values
df['dual_meet_matches'] = df.groupby('wrestler_id')['is_dual_meet'].transform(
    lambda x: x.cumsum().shift(1).fillna(0)
)
df['dual_meet_win_rate'] = np.where(df['dual_meet_matches'] > 0,
                                    df['dual_meet_wins'] / df['dual_meet_matches'], 0.5)

# Tournament performance
df['tournament_wins'] = df.groupby('wrestler_id')[['is_win', 'is_dual_meet']].apply(
    lambda x: (x['is_win'] * (1 - x['is_dual_meet'])).cumsum().shift(1).fillna(0)
).values
df['tournament_matches'] = df.groupby('wrestler_id')['is_dual_meet'].apply(
    lambda x: (1 - x).cumsum().shift(1).fillna(0)
).values
df['tournament_win_rate'] = np.where(df['tournament_matches'] > 0,
                                    df['tournament_wins'] / df['tournament_matches'], 0.5)

In [14]:
df[(df['wrestler'] == 'Jacori Teemer')].head(25)

,season,date,event,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,...,overtime_rate_last_5,overtime_rate_last_10,avg_duration_last_5,avg_duration_last_10,dual_meet_wins,dual_meet_matches,dual_meet_win_rate,tournament_wins,tournament_matches,tournament_win_rate
268337,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Brayton Lee,52079,Minnesota,...,NaN,NaN,NaN,NaN,0.0,0.0,0.500000,0.0,0.0,0.500000
268338,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Sammy Sasso,51143,Ohio State,...,0.000000,0.000000,420.0,420.000000,0.0,0.0,0.500000,1.0,1.0,1.000000
268339,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Matt Kolodzik,45071,Princeton,...,0.000000,0.000000,420.0,420.000000,0.0,0.0,0.500000,2.0,2.0,1.000000
268340,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Patricio Lugo,39919,Iowa,...,0.333333,0.333333,460.0,460.000000,0.0,0.0,0.500000,2.0,3.0,0.666667
268341,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Alfred Bannister,19997,Maryland,...,0.250000,0.250000,450.0,450.000000,0.0,0.0,0.500000,2.0,4.0,0.500000
268342,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Khristian Olivas,40318,Fresno State,...,0.200000,0.200000,444.0,444.000000,0.0,0.0,0.500000,3.0,5.0,0.600000
268343,2018/2019,2018-12-29,Midlands Championships,149,Jacori Teemer,50581,Arizona State,Jake Benner,50617,Rutgers,...,0.200000,0.166667,444.0,440.000000,0.0,0.0,0.500000,4.0,6.0,0.666667
268344,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Tyler Eischens,51205,Stanford,...,0.200000,0.142857,444.0,437.142857,0.0,0.0,0.500000,5.0,7.0,0.714286
268345,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Justin McCoy,50946,Virginia,...,0.000000,0.125000,420.0,435.000000,0.0,0.0,0.500000,6.0,8.0,0.750000
268346,2018/2019,2019-03-02,National Collegiate Open,157,Jacori Teemer,50581,Arizona State,Seth Hogue,44668,West Virginia,...,0.000000,0.111111,420.0,433.333333,0.0,0.0,0.500000,7.0,9.0,0.777778


In [15]:
# Opponent Strength Features
# Calculate each wrestler's overall performance for opponent strength metrics
wrestler_performance = df.groupby('wrestler_id').agg({
    'is_win': ['count', 'sum'],
    'wrestler_score': 'mean',
    'opponent_score': 'mean',
    'bonus_win': 'mean'
}).reset_index()

wrestler_performance.columns = ['wrestler_id', 'total_matches', 'total_wins', 
                                'avg_score', 'avg_allowed', 'bonus_rate']
wrestler_performance['overall_win_rate'] = wrestler_performance['total_wins'] / wrestler_performance['total_matches']

# Merge opponent stats
df = df.merge(wrestler_performance[['wrestler_id', 'overall_win_rate', 'avg_score', 'bonus_rate']], 
                left_on='opponent_id', right_on='wrestler_id', 
                how='left', suffixes=('', '_opp'))
df = df.drop('wrestler_id_opp', axis=1)
df = df.rename(columns={'overall_win_rate': 'opponent_career_win_rate',
                        'avg_score': 'opponent_avg_score',
                        'bonus_rate': 'opponent_bonus_rate'})

# Rolling opponent strength
for window in [3, 5]:
    df[f'avg_opponent_win_rate_last_{window}'] = df.groupby('wrestler_id')['opponent_career_win_rate'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )

# Quality opponent record (opponents with >65% win rate)
df['quality_opponent'] = (df['opponent_career_win_rate'] > 0.65).astype(int)
for window in [5, 10]:
    df[f'quality_opponent_win_rate_last_{window}'] = df.groupby('wrestler_id')[['is_win', 'quality_opponent']].apply(
        lambda x: (x['is_win'] * x['quality_opponent']).rolling(window=window, min_periods=1).sum().shift(1) /
                    x['quality_opponent'].rolling(window=window, min_periods=1).sum().shift(1)
    ).fillna(0.5).values

In [16]:
df[(df['wrestler'] == 'Jacori Teemer')].head(25)[['wrestler', 'opponent', 'opponent_career_win_rate', 'avg_opponent_win_rate_last_3', 'avg_opponent_win_rate_last_5', 'quality_opponent', 'is_win', 'quality_opponent_win_rate_last_5', 'quality_opponent_win_rate_last_10']]

,wrestler,opponent,opponent_career_win_rate,avg_opponent_win_rate_last_3,avg_opponent_win_rate_last_5,quality_opponent,is_win,quality_opponent_win_rate_last_5,quality_opponent_win_rate_last_10
268337,Jacori Teemer,Brayton Lee,0.746032,NaN,NaN,1,1,0.500000,0.500000
268338,Jacori Teemer,Sammy Sasso,0.859155,0.746032,0.746032,1,1,1.000000,1.000000
268339,Jacori Teemer,Matt Kolodzik,0.860294,0.802593,0.802593,1,0,1.000000,1.000000
268340,Jacori Teemer,Patricio Lugo,0.780142,0.821827,0.821827,1,0,0.666667,0.666667
268341,Jacori Teemer,Alfred Bannister,0.703125,0.833197,0.811406,1,1,0.500000,0.500000
268342,Jacori Teemer,Khristian Olivas,0.676923,0.781187,0.789750,1,1,0.600000,0.600000
268343,Jacori Teemer,Jake Benner,0.435897,0.720063,0.775928,0,1,0.600000,0.666667
268344,Jacori Teemer,Tyler Eischens,0.673469,0.605315,0.691276,1,1,0.500000,0.666667
268345,Jacori Teemer,Justin McCoy,0.746479,0.595430,0.653911,1,1,0.750000,0.714286
268346,Jacori Teemer,Seth Hogue,0.410000,0.618615,0.647179,0,1,1.000000,0.750000


In [17]:
from datetime import timedelta, datetime

In [18]:
def calculate_match_frequency(dates, window_days=30):
    """Calculate matches per week over rolling window"""
    frequencies = []
    
    for i in range(len(dates)):
        if i == 0:
            frequencies.append(0)
        else:
            window_start = dates.iloc[i] - timedelta(days=window_days)
            recent_matches = sum(1 for j in range(i) if dates.iloc[j] >= window_start)
            matches_per_week = recent_matches * 7 / window_days
            frequencies.append(matches_per_week)
    
    return pd.Series(frequencies)

In [19]:
df['date'] = pd.to_datetime(df['date'])
df['date']

0        2013-11-22
1        2013-11-22
2        2013-12-01
3        2013-12-05
4        2013-12-12
            ...    
506467   2025-03-02
506468   2022-11-13
506469   2022-11-13
506470   2022-11-13
506471   2022-11-13
Name: date, Length: 506472, dtype: datetime64[ns]

In [20]:
# Time Based Features
df['days_since_last_match'] = df.groupby('wrestler_id')['date'].diff().dt.days
df['days_since_last_match'] = df['days_since_last_match'].fillna(0)

# Match frequency (matches per week over rolling periods)
df['matches_per_week_last_30_days'] = df.groupby('wrestler_id')['date'].apply(
    lambda x: calculate_match_frequency(x.reset_index(drop=True), window_days=30)
).values

# Yearly performance
df['year'] = df['date'].dt.year
yearly_performance = df.groupby(['wrestler_id', 'year'])['is_win'].mean().reset_index()
yearly_performance['year'] = yearly_performance['year'] + 1
yearly_performance.columns = ['wrestler_id', 'year', 'prev_yearly_win_rate']
df = df.merge(yearly_performance, on=['wrestler_id', 'year'], how='left')
df['prev_yearly_win_rate'] = df['prev_yearly_win_rate'].fillna(0.5)

In [21]:
df[(df['wrestler'] == 'Jacori Teemer')].head(25)[['date', 'days_since_last_match', 'matches_per_week_last_30_days', 'prev_yearly_win_rate']]

,date,days_since_last_match,matches_per_week_last_30_days,prev_yearly_win_rate
268337,2018-12-29,0.0,0.000000,0.500000
268338,2018-12-29,0.0,0.233333,0.500000
268339,2018-12-29,0.0,0.466667,0.500000
268340,2018-12-29,0.0,0.700000,0.500000
268341,2018-12-29,0.0,0.933333,0.500000
268342,2018-12-29,0.0,1.166667,0.500000
268343,2018-12-29,0.0,1.400000,0.500000
268344,2019-03-02,63.0,0.000000,0.714286
268345,2019-03-02,0.0,0.233333,0.714286
268346,2019-03-02,0.0,0.466667,0.714286


In [22]:
df = df.sort_values(['wrestler_id', 'date'])

# Shift to get experience before current match
df['experience'] = df.groupby('wrestler_id').cumcount().fillna(0)

# Map opponent experience
exp_mapping = df.groupby(['wrestler_id', 'date'])['experience'].first().reset_index()

df = df.merge(
    exp_mapping.rename(columns={'wrestler_id': 'opponent_id', 'experience': 'opponent_experience'}),
    on=['opponent_id', 'date'],
    how='left'
)

# Calculate experience differential
df['experience_differential'] = df['experience'] - df['opponent_experience']

In [23]:
df[(df['wrestler'] == 'Jacori Teemer')].head(25)[['experience', 'opponent_experience', 'experience_differential']]

,experience,opponent_experience,experience_differential
268337,0,14,-14
268338,1,7,-6
268339,2,99,-97
268340,3,93,-90
268341,4,112,-108
268342,5,45,-40
268343,6,7,-1
268344,7,18,-11
268345,8,21,-13
268346,9,73,-64


In [24]:
for window in [5, 10]:
    # Map opponent experience
    rate_mapping = df.groupby(['wrestler_id', 'date'])[f'win_rate_last_{window}'].first().reset_index()

    df = df.merge(
        rate_mapping.rename(columns={'wrestler_id': 'opponent_id', f'win_rate_last_{window}': f'opponent_win_rate_last_{window}'}),
        on=['opponent_id', 'date'],
        how='left'
    )

    # Calculate experience differential
    df[f'form_differential_{window}'] = df[f'win_rate_last_{window}'] - df[f'opponent_win_rate_last_{window}']


In [25]:
# Map opponent scoring rate
point_mapping = df.groupby(['wrestler_id', 'date'])['avg_points_scored_last_5'].first().reset_index()

df = df.merge(
    point_mapping.rename(columns={'wrestler_id': 'opponent_id', 'avg_points_scored_last_5': 'opponent_avg_points_scored_last_5'}),
    on=['opponent_id', 'date'],
    how='left'
)

# Calculate scoring rate differential
df['scoring_advantage_last_5'] = df['avg_points_scored_last_5'] - df['opponent_avg_points_scored_last_5']

# Map opponent scoring allowed rate
point_mapping = df.groupby(['wrestler_id', 'date'])['avg_points_allowed_last_5'].first().reset_index()

df = df.merge(
    point_mapping.rename(columns={'wrestler_id': 'opponent_id', 'avg_points_allowed_last_5': 'opponent_avg_points_allowed_last_5'}),
    on=['opponent_id', 'date'],
    how='left'
)

# Calculate scoring rate differential
df['defensive_advantage_last_5'] = df['avg_points_allowed_last_5'] - df['opponent_avg_points_allowed_last_5']

In [26]:
df[(df['wrestler'] == 'Jacori Teemer')].head(25)[['wrestler', 'opponent', 'wrestler_score', 'opponent_score', 'avg_points_scored_last_5', 'opponent_avg_points_scored_last_5', 'scoring_advantage_last_5', 'avg_points_allowed_last_5', 'opponent_avg_points_allowed_last_5', 'defensive_advantage_last_5']]

,wrestler,opponent,wrestler_score,opponent_score,avg_points_scored_last_5,opponent_avg_points_scored_last_5,scoring_advantage_last_5,avg_points_allowed_last_5,opponent_avg_points_allowed_last_5,defensive_advantage_last_5
268337,Jacori Teemer,Brayton Lee,9,7,NaN,12.0,NaN,NaN,5.6,NaN
268338,Jacori Teemer,Sammy Sasso,8,6,9.0,10.8,-1.8,7.000000,3.8,3.200000
268339,Jacori Teemer,Matt Kolodzik,4,6,8.5,13.8,-5.3,6.500000,3.0,3.500000
268340,Jacori Teemer,Patricio Lugo,3,10,7.0,5.8,1.2,6.333333,5.4,0.933333
268341,Jacori Teemer,Alfred Bannister,9,5,6.0,4.2,1.8,7.250000,3.4,3.850000
268342,Jacori Teemer,Khristian Olivas,11,2,6.6,6.0,0.6,6.800000,2.4,4.400000
268343,Jacori Teemer,Jake Benner,8,5,7.0,5.0,2.0,5.800000,6.4,-0.600000
268344,Jacori Teemer,Tyler Eischens,3,2,7.0,6.6,0.4,5.600000,6.2,-0.600000
268345,Jacori Teemer,Justin McCoy,7,5,6.8,5.4,1.4,4.800000,4.4,0.400000
268346,Jacori Teemer,Seth Hogue,0,0,7.6,3.8,3.8,3.800000,3.6,0.200000


In [27]:
# Map opponent rest
rest_mapping = df.groupby(['wrestler_id', 'date'])['days_since_last_match'].first().reset_index()

df = df.merge(
    rest_mapping.rename(columns={'wrestler_id': 'opponent_id', 'days_since_last_match': 'opponent_days_since_last_match'}),
    on=['opponent_id', 'date'],
    how='left'
)

# Calculate rest differential
df['rest_differential'] = df['days_since_last_match'] - df['opponent_days_since_last_match']

In [28]:
df[(df['wrestler'] == 'Jacori Teemer')].iloc[100:125][['date', 'days_since_last_match', 'opponent_days_since_last_match', 'rest_differential']]

,date,days_since_last_match,opponent_days_since_last_match,rest_differential
268437,2024-03-21,0.0,12.0,-12.0
268438,2024-03-21,0.0,11.0,-11.0
268439,2024-11-09,233.0,1.0,232.0
268440,2024-11-15,6.0,12.0,-6.0
268441,2025-01-25,71.0,6.0,65.0
268442,2025-01-31,6.0,7.0,-1.0
268443,2025-02-14,14.0,5.0,9.0
268444,2025-02-23,9.0,15.0,-6.0
268445,2025-03-08,13.0,35.0,-22.0
268446,2025-03-08,0.0,15.0,-15.0


In [29]:
def calculate_h2h_record(df, wrestler_id, opponent_id, current_date):
    """Calculate head-to-head record between two wrestlers before current date"""
    # Get all matches between these two wrestlers before current date
    h2h_matches = df[
        (((df['wrestler_id'] == wrestler_id) & (df['opponent_id'] == opponent_id)) |
         ((df['wrestler_id'] == opponent_id) & (df['opponent_id'] == wrestler_id))) &
        (df['date'] < current_date)
    ]
    
    if len(h2h_matches) == 0:
        return 0, 0
    
    # Count wins for the wrestler in question
    wrestler_wins = len(h2h_matches[
        (h2h_matches['wrestler_id'] == wrestler_id) & (h2h_matches['result'] == 'W')
    ])
    
    return wrestler_wins, len(h2h_matches)

In [30]:
# Head-to-head record
# Create head-to-head combinations
df['h2h_key'] = df.apply(lambda x: tuple(sorted([x['wrestler_id'], x['opponent_id']])), axis=1)

# Calculate cumulative H2H record
df['h2h_matches'] = df.groupby(['wrestler_id', 'opponent_id']).cumcount()
df['h2h_wins'] = df.groupby(['wrestler_id', 'opponent_id'])['is_win'].cumsum()

# Calculate H2H win rate (shift to avoid data leakage)
df['h2h_win_rate'] = df['h2h_wins'] / (df['h2h_matches'] + 1)
df['h2h_win_rate'] = df.groupby(['wrestler_id', 'opponent_id'])['h2h_win_rate'].shift(1)

# Fill NaN with 0.5 (no previous history)
df['h2h_win_rate'] = df['h2h_win_rate'].fillna(0.5)
df.loc[df['is_win'] == True, 'h2h_wins'] = df['h2h_wins'] - 1

In [31]:
df[((df['wrestler'] == 'Beau Bartlett') & (df['opponent'] == 'Jesse Mendez')) | 
   ((df['opponent'] == 'Beau Bartlett') & (df['wrestler'] == 'Jesse Mendez'))][['wrestler', 'opponent', 'is_win', 'h2h_matches', 'h2h_wins', 'h2h_win_rate']]

,wrestler,opponent,is_win,h2h_matches,h2h_wins,h2h_win_rate
356584,Beau Bartlett,Jesse Mendez,1,0,0,0.500000
356589,Beau Bartlett,Jesse Mendez,0,1,1,1.000000
356592,Beau Bartlett,Jesse Mendez,0,2,1,0.500000
356612,Beau Bartlett,Jesse Mendez,1,3,1,0.333333
356615,Beau Bartlett,Jesse Mendez,1,4,2,0.500000
356619,Beau Bartlett,Jesse Mendez,0,5,3,0.600000
442950,Jesse Mendez,Beau Bartlett,0,0,0,0.500000
442956,Jesse Mendez,Beau Bartlett,1,1,0,0.000000
442961,Jesse Mendez,Beau Bartlett,1,2,1,0.500000
442985,Jesse Mendez,Beau Bartlett,0,3,2,0.666667


In [32]:
# Weight class experience
df['weight_class_matches'] = df.groupby(['wrestler_id', 'weight_class']).cumcount()
df['weight_class_wins'] = df.groupby(['wrestler_id', 'weight_class'])['is_win'].cumsum().shift(1).fillna(0)
df.loc[df['weight_class_matches'] == 0, 'weight_class_wins'] = 0
df['weight_class_win_rate'] = np.where(df['weight_class_matches'] > 0,
                                        df['weight_class_wins'] / df['weight_class_matches'], 0.5)

In [33]:
df[(df['wrestler'] == 'Jacori Teemer')][['weight_class', 'weight_class_matches', 'is_win', 'weight_class_wins', 'weight_class_win_rate']]

,weight_class,weight_class_matches,is_win,weight_class_wins,weight_class_win_rate
268337,149,0,1,0.0,0.500000
268338,149,1,1,1.0,1.000000
268339,149,2,0,2.0,1.000000
268340,149,3,0,2.0,0.666667
268341,149,4,1,2.0,0.500000
...,...,...,...,...,...
268447,157,103,0,88.0,0.854369
268448,157,104,1,88.0,0.846154
268449,157,105,0,89.0,0.847619
268450,157,106,0,89.0,0.839623


In [34]:
df.columns

Index(['season', 'date', 'event', 'weight_class', 'wrestler', 'wrestler_id',
       'wrestler_school', 'opponent', 'opponent_id', 'opponent_school',
       'result', 'result_type', 'wrestler_score', 'opponent_score',
       'is_dual_meet', 'duration_seconds', 'is_overtime', 'is_win',
       'point_differential', 'win_rate_last_3', 'win_rate_last_5',
       'win_rate_last_10', 'win_rate_last_15', 'streak', 'career_wins',
       'career_losses', 'career_matches', 'season_wins', 'season_matches',
       'season_win_rate', 'bonus_win', 'close_match', 'bonus_win_rate_last_5',
       'close_match_win', 'close_match_win_rate_last_5',
       'bonus_win_rate_last_10', 'close_match_win_rate_last_10',
       'avg_points_scored_last_3', 'avg_points_allowed_last_3',
       'avg_point_differential_last_3', 'avg_points_scored_last_5',
       'avg_points_allowed_last_5', 'avg_point_differential_last_5',
       'avg_points_scored_last_10', 'avg_points_allowed_last_10',
       'avg_point_differential_la

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506472 entries, 0 to 506471
Data columns (total 88 columns):
 #   Column                              Non-Null Count   Dtype         
---  ------                              --------------   -----         
 0   season                              506472 non-null  object        
 1   date                                506472 non-null  datetime64[ns]
 2   event                               506472 non-null  object        
 3   weight_class                        506472 non-null  int64         
 4   wrestler                            506472 non-null  object        
 5   wrestler_id                         506472 non-null  int64         
 6   wrestler_school                     506472 non-null  object        
 7   opponent                            506472 non-null  object        
 8   opponent_id                         506472 non-null  int64         
 9   opponent_school                     506472 non-null  object        
 10  result  

In [37]:
nan_counts = df.isnull().sum()
nan_columns_with_counts = nan_counts[nan_counts > 0]
nan_columns_with_counts

win_rate_last_3                       33330
win_rate_last_5                       33330
win_rate_last_10                      33330
win_rate_last_15                      33330
bonus_win_rate_last_5                 33330
bonus_win_rate_last_10                33330
avg_points_scored_last_3              33330
avg_points_allowed_last_3             33330
avg_point_differential_last_3         33330
avg_points_scored_last_5              33330
avg_points_allowed_last_5             33330
avg_point_differential_last_5         33330
avg_points_scored_last_10             33330
avg_points_allowed_last_10            33330
avg_point_differential_last_10        33330
overtime_rate_last_5                  33330
overtime_rate_last_10                 33330
avg_duration_last_5                   33330
avg_duration_last_10                  33330
avg_opponent_win_rate_last_3          33330
avg_opponent_win_rate_last_5          33330
opponent_win_rate_last_5              18591
form_differential_5             

In [41]:
50400/len(df) * 100

9.951191773681467

In [101]:
def prepare_data_for_modeling(df):
    """Prepare data for logistic regression modeling"""
    non_feature_cols = [
        'duration_seconds', 'point_differential',
        'wrestler_score', 'opponent_score', 
        'result', 'result_type', 'is_overtime', 
        'bonus_win', 'close_match', 'h2h_key',
        'season', 'year', 'wrestler', 'opponent', 
        'event', 'wrestler_school', 'opponent_school'
    ]

    priority_features = [col for col in df.columns if col not in non_feature_cols + ['is_win', 'date']]
    
    # Remove rows with NaN values in priority features
    df_clean = df.dropna(subset=priority_features + ['is_win'])
    
    # Separate features and target, keep date column
    X = df_clean[priority_features + ['date']]
    y = df_clean['is_win']
    
    return X, y, priority_features

In [102]:
def temporal_train_test_split(X, y, test_size=0.2, split_date=None):
    """Split data based on time to avoid data leakage"""
    
    if split_date is None:
        # Calculate split date based on test_size
        sorted_dates = X['date'].sort_values()
        split_idx = int(len(sorted_dates) * (1 - test_size))
        split_date = sorted_dates.iloc[split_idx]
    
    # Create temporal split
    train_mask = X['date'] < split_date
    test_mask = X['date'] >= split_date
    
    X_train = X[train_mask]
    X_test = X[test_mask]
    y_train = y[train_mask]
    y_test = y[test_mask]
    
    print(f"Split date: {split_date}")
    print(f"Train set: {len(X_train)} matches ({X_train['date'].min()} to {X_train['date'].max()})")
    print(f"Test set: {len(X_test)} matches ({X_test['date'].min()} to {X_test['date'].max()})")
    
    return X_train, X_test, y_train, y_test, split_date

In [103]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text
from sklearn.metrics import accuracy_score, classification_report

In [104]:
def train_decision_tree(X, y, feature_columns, max_depth=10, min_samples_split=20, min_samples_leaf=10):
    
    # Temporal split
    X_train, X_test, y_train, y_test, split_date = temporal_train_test_split(X, y)
    
    # Decision trees don't require feature scaling, but we'll keep dates separate
    X_train_features = X_train[feature_columns]
    X_test_features = X_test[feature_columns]
    
    # Train model
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    model.fit(X_train_features, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_features)
    y_pred_proba = model.predict_proba(X_test_features)[:, 1]
    
    # Print results
    print("\nDecision Tree Results:")
    print("=" * 50)
    print(f"Tree depth: {model.tree_.max_depth}")
    print(f"Number of leaves: {model.tree_.n_leaves}")
    print(f"Training accuracy: {accuracy_score(y_train, model.predict(X_train_features)):.4f}")
    print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Feature importance
    print("\nFeature Importance:")
    print("=" * 50)
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance)
    
    # Print simplified tree rules (first few levels only)
    print(f"\nDecision Tree Rules (max depth {min(3, max_depth)}):")
    print("=" * 50)
    tree_rules = export_text(model, feature_names=feature_columns, max_depth=3)
    print(tree_rules)
    
    return model, X_test, y_test, y_pred, y_pred_proba, feature_importance, split_date

In [105]:
#df = engineer_all_features(raw).drop(columns=['index'])
X, y, feature_cols = prepare_data_for_modeling(df)
model, X_test, y_test, y_pred, y_pred_proba, feature_importance, split_date = train_decision_tree(X, y, feature_cols)

Split date: 2023-02-10 00:00:00
Train set: 364637 matches (2013-11-02 00:00:00 to 2023-02-09 00:00:00)
Test set: 91435 matches (2023-02-10 00:00:00 to 2025-03-20 00:00:00)

Decision Tree Results:
Tree depth: 10
Number of leaves: 466
Training accuracy: 0.8714
Test accuracy: 0.8458

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85     45380
           1       0.89      0.79      0.84     46055

    accuracy                           0.85     91435
   macro avg       0.85      0.85      0.85     91435
weighted avg       0.85      0.85      0.85     91435


Feature Importance:
                              feature  importance
16                    close_match_win    0.531154
56               form_differential_10    0.177598
39           opponent_career_win_rate    0.125156
35                 dual_meet_win_rate    0.028757
38                tournament_win_rate    0.026107
..                                ...         .

In [118]:
combined = pd.concat([X_test[['date', 'wrestler_id', 'opponent_id']], pd.DataFrame({'is_win': y_test, 'predicted_win': y_pred, 'predicted_win_proba': y_pred_proba})], axis=1)

In [119]:
combined.loc[combined['is_win'] != combined['predicted_win'], :].sort_values('predicted_win_proba', ascending=False)

,date,wrestler_id,opponent_id,is_win,predicted_win,predicted_win_proba
493445,2024-12-20,85062,85079,0,1,0.988294
446265,2023-11-11,75034,79412,0,1,0.988294
397758,2024-12-21,68738,71941,0,1,0.986530
411252,2025-02-14,71751,78773,0,1,0.986530
324279,2023-02-19,57070,72683,0,1,0.986530
...,...,...,...,...,...,...
414880,2024-02-04,71870,61129,1,0,0.003519
480049,2024-11-23,82489,79603,1,0,0.003519
450645,2025-01-25,76673,93942,1,0,0.003519
426051,2023-11-11,72913,84949,1,0,0.000000
